# 02 实验目标与统计精度分析

**说明：MDE 与 δ 在分析开始前锁定（2026-08-31），本 Notebook 从 `config/analysis_config.toml` 读取，不就地修改。**

- 统计假设、参数锁定、baseline 来源、理论样本量与周期、实际样本与检测能力、配置集中化。
- 本 Notebook 只评估“实验能检测多大效应”（精度/功效），**不做处理效应是否显著的结论**。
- 不使用 misleading 的 post-hoc observed power；功效曲线是“在已实现样本量下，对不同**假设真实效应**的检测能力”，属设计式精度分析。

In [1]:
# 加载锁定配置与清洗产物
import json
import tomllib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
with open(ROOT / "config" / "analysis_config.toml", "rb") as f:
    CFG = tomllib.load(f)
LK, BASE, EXP = CFG["locked"], CFG["design_baseline"], CFG["exploratory"]
pooled = pd.read_csv(ROOT / "data/processed/pooled_summary.csv").set_index("Group")
long = pd.read_csv(ROOT / "data/processed/daily_long.csv", parse_dates=["Date"])
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
print("alpha(two-sided)=", LK["alpha_two_sided"], "| NI one-sided=", LK["ni_alpha_one_sided"],
      "| target power=", LK["target_power"])
print("MDE gross/net=", LK["mde_gross"], LK["mde_net"], "| NI delta=", LK["ni_delta_net"])
pooled

alpha(two-sided)= 0.05 | NI one-sided= 0.025 | target power= 0.8
MDE gross/net= 0.01 0.0075 | NI delta= 0.0075


,Pageviews_37d,Clicks_37d,CTP_37d,Outcome_days,Clicks_23d,Enrollments_23d,Payments_23d,GrossConversion,NetConversion,PayPerEnrollment
Group,,,,,,,,,,
Control,345543,28378,0.082126,23,17293,3785,2033,0.218875,0.117562,0.537120
Experiment,344660,28325,0.082182,23,17260,3423,1945,0.198320,0.112688,0.568215


## 统计假设
- **Gross / Net（双侧）**：H0: Δ=0 vs H1: Δ≠0，Δ = Experiment − Control，α=0.05。
- **Net 非劣效**：H0: Δ ≤ −δ vs H1: Δ > −δ；单侧 α=0.025，用 95% 双侧 CI 下界 > −δ 判定。
- 普通显著性与非劣效两套 α 口径分开，不混用。

## baseline 来源（优先级 2：公开 benchmark）
数据集无实验前历史 → planning baseline 采用 Udacity 课程公开规划假设；observed control rate 仅用于事后精度。

In [2]:
# 目标 MDE 下理论所需样本（标准两比例公式，等组，双侧）
def n_per_group(p1, d, alpha, power):
    p2 = p1 + d; pbar = (p1 + p2) / 2
    za, zb = norm.ppf(1 - alpha/2), norm.ppf(power)
    return ((za*np.sqrt(2*pbar*(1-pbar)) + zb*np.sqrt(p1*(1-p1)+p2*(1-p2)))**2) / (p1-p2)**2

a, pw = LK["alpha_two_sided"], LK["target_power"]
plan = []
for name, p1, mde in [("Gross", BASE["p_enroll_given_click"], LK["mde_gross"]),
                      ("Net", BASE["p_pay_given_click"], LK["mde_net"])]:
    n8 = n_per_group(p1, -mde, a, pw)
    n9 = n_per_group(p1, -mde, a, 0.90)
    pv_total = 2 * n8 / BASE["ctr"]
    accrual_days = 2*n8 / BASE["daily_clicks_total"]
    plan.append({"Metric": name, "planning_p": p1, "target_MDE": mde,
                 "n_per_group@0.80": round(n8), "n_per_group@0.90": round(n9),
                 "total_PV@0.80": round(pv_total),
                 "accrual_days": round(accrual_days, 1),
                 "calendar_days_with_14d_lag": round(accrual_days + CFG["locked"]["windows"]["outcome_lag_days"], 1)})
plan_df = pd.DataFrame(plan)
# 与 Udacity 课程官方解逐指标比对（差异来自零假设方差/取整约定；不使用笼统阈值）
COURSE_OFFICIAL = {"Gross": 25835, "Net": 27413}
for r in plan:
    ours, off = r["n_per_group@0.80"], COURSE_OFFICIAL[r["Metric"]]
    gap = (off - ours) / off
    print(f"{r['Metric']}: ours={ours:,} vs course-official={off:,} -> gap {gap:.1%}")
print("差异源于零假设方差(pooled pbar vs baseline p1)与 z 值取整约定；量级一致，underpowered 结论不受影响。")
plan_df

Gross: ours=25,233 vs course-official=25,835 -> gap 2.3%
Net: ours=26,348 vs course-official=27,413 -> gap 3.9%
差异源于零假设方差(pooled pbar vs baseline p1)与 z 值取整约定；量级一致，underpowered 结论不受影响。


,Metric,planning_p,target_MDE,n_per_group@0.80,n_per_group@0.90,total_PV@0.80,accrual_days,calendar_days_with_14d_lag
0,Gross,0.206250,0.0100,25233,33779,630818,15.8,29.8
1,Net,0.109313,0.0075,26348,35273,658712,16.5,30.5


In [3]:
# 实际样本、实际估计 SE 与 95% CI 半宽（只谈精度，不做判定）
ow = long[long["OutcomeComplete"]]
actual = {}
for name, num, den in [("Gross", "Enrollments", "Clicks"), ("Net", "Payments", "Clicks")]:
    d = ow.groupby("Group").apply(lambda x: pd.Series({
        "n": int(x[den].sum()), "x": int(x[num].sum()),
        "p": x[num].sum()/x[den].sum()}), include_groups=False)
    nc, ne = d.loc["Control", "n"], d.loc["Experiment", "n"]
    pc, pe = d.loc["Control", "p"], d.loc["Experiment", "p"]
    se = np.sqrt(pc*(1-pc)/nc + pe*(1-pe)/ne)          # unpooled，CI 口径
    hw = norm.ppf(1-a/2)*se
    actual[name] = dict(nc=int(nc), ne=int(ne), pc=pc, pe=pe, se=se, ci_halfwidth=hw, ci_fullwidth=2*hw)
    print(f"{name:5s}: n_c={int(nc):,}, n_e={int(ne):,}, p_c={pc:.6f}, p_e={pe:.6f}")
    print(f"       unpooled SE={se:.6f}, 95% CI half-width={hw:.6f} ({hw*100:.3f}pp), full width={2*hw*100:.3f}pp")
print("有效 outcome 天数 =", int(ow["Date"].nunique()), "；位置/显著性判定留待后续分析。")

Gross: n_c=17,293, n_e=17,260, p_c=0.218875, p_e=0.198320
       unpooled SE=0.004370, 95% CI half-width=0.008565 (0.857pp), full width=1.713pp
Net  : n_c=17,293, n_e=17,260, p_c=0.117562, p_e=0.112688
       unpooled SE=0.003434, 95% CI half-width=0.006730 (0.673pp), full width=1.346pp
有效 outcome 天数 = 23 ；位置/显著性判定留待后续分析。


In [4]:
# 在已实现样本量下，对不同【假设真实效应】的检测能力（设计式功效，非 observed power）
def power_two_prop(p, d, nc, ne, alpha=a):
    # p=对照真实率(规划基线), d=真实处理效应(可为负), 两组实际样本量
    p0 = (nc*p + ne*(p+d))/(nc+ne)
    se0 = np.sqrt(p0*(1-p0)*(1/nc + 1/ne))
    seA = np.sqrt(p*(1-p)/nc + (p+d)*(1-(p+d))/ne)
    zc = norm.ppf(1-alpha/2)*se0
    return float(norm.sf((zc-d)/seA) + norm.cdf((-zc-d)/seA))

from scipy.optimize import brentq
def achievable_mde(p, nc, ne, power):
    f = lambda x: power_two_prop(p, -x, nc, ne) - power
    # 上界 p-eps：处理后率 p-x 必须为正，避免对负数开方
    return brentq(f, 1e-6, p - 1e-6)

rows = []
grid = [0.005, 0.0075, 0.01, 0.0125, 0.015, 0.02]
for name, p1, mde in [("Gross", BASE["p_enroll_given_click"], LK["mde_gross"]),
                      ("Net", BASE["p_pay_given_click"], LK["mde_net"])]:
    nc, ne = actual[name]["nc"], actual[name]["ne"]
    p_lock = power_two_prop(p1, -mde, nc, ne)
    mde80 = achievable_mde(p1, nc, ne, LK["target_power"])
    req = n_per_group(p1, -mde, a, LK["target_power"])
    rows.append({"Metric": name, "power@locked_MDE": round(p_lock, 4),
                 "achievable_MDE@0.80": round(mde80, 5),
                 "locked_MDE": mde, "n_coverage": round((nc+ne)/2/req, 3)})
    print(f"{name}: 实际样本对锁定 MDE({mde:.2%}) 的功效={p_lock:.3f}；"
          f"要达到 80% 功效，实际可检测 MDE≈{mde80*100:.3f}pp（锁定值 {mde:.2%}）；样本覆盖率≈{(nc+ne)/2/req:.1%}")
    print("  不同假设真实效应下的功效：",
          {f"{x*100:.2f}pp": round(power_two_prop(p1, -x, nc, ne), 3) for x in grid})
power_tbl = pd.DataFrame(rows)

Gross: 实际样本对锁定 MDE(1.00%) 的功效=0.640；要达到 80% 功效，实际可检测 MDE≈1.206pp（锁定值 1.00%）；样本覆盖率≈68.5%
  不同假设真实效应下的功效： {'0.50pp': 0.211, '0.75pp': 0.411, '1.00pp': 0.64, '1.25pp': 0.828, '1.50pp': 0.937, '2.00pp': 0.997}
Net: 实际样本对锁定 MDE(0.75%) 的功效=0.621；要达到 80% 功效，实际可检测 MDE≈0.923pp（锁定值 0.75%）；样本覆盖率≈65.6%
  不同假设真实效应下的功效： {'0.50pp': 0.325, '0.75pp': 0.621, '1.00pp': 0.86, '1.25pp': 0.969, '1.50pp': 0.996, '2.00pp': 1.0}


In [5]:
# 图1：功效 vs 假设真实效应（实际样本量）
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
xs = np.linspace(0.001, 0.02, 200)
for name, p1, mde, color in [("Gross", BASE["p_enroll_given_click"], LK["mde_gross"], "#1f77b4"),
                             ("Net", BASE["p_pay_given_click"], LK["mde_net"], "#d62728")]:
    nc, ne = actual[name]["nc"], actual[name]["ne"]
    ys = [power_two_prop(p1, -x, nc, ne) for x in xs]
    ax.plot(xs*100, ys, color=color, label=name)
    ax.axvline(mde*100, color=color, ls="--", lw=1)
ax.axhline(0.8, color="gray", ls=":", lw=1.2)
ax.text(xs[-1]*100, 0.81, "target power 0.80", ha="right", color="gray")
ax.set_xlabel("Hypothesized true absolute effect (percentage points)")
ax.set_ylabel("Power (two-sided, alpha=0.05)")
ax.set_title("Power vs effect size at achieved sample size")
ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIG / "fig_power_vs_effect.png"); plt.close(fig)
print("saved reports/figures/fig_power_vs_effect.png")

saved reports/figures/fig_power_vs_effect.png

In [6]:
# 图2：80% 功效所需 MDE vs 每组样本量（标注实际样本与锁定 MDE）
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
ns = np.arange(8000, 40001, 500)
for name, p1, mde, color in [("Gross", BASE["p_enroll_given_click"], LK["mde_gross"], "#1f77b4"),
                             ("Net", BASE["p_pay_given_click"], LK["mde_net"], "#d62728")]:
    ys = [achievable_mde(p1, n, n, LK["target_power"])*100 for n in ns]
    ax.plot(ns, ys, color=color, label=name)
    ax.axhline(mde*100, color=color, ls="--", lw=1)
n_act = int((actual["Gross"]["nc"]+actual["Gross"]["ne"])/2)
ax.axvline(n_act, color="black", ls=":", lw=1.2)
ax.text(n_act+300, ax.get_ylim()[1]*0.9, f"achieved ~{n_act:,}/group", rotation=90, va="top")
ax.set_xlabel("Clicks per group"); ax.set_ylabel("Detectable |MDE| at 80% power (pp)")
ax.set_title("Precision (achievable MDE) vs sample size")
ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIG / "fig_mde_vs_sample.png"); plt.close(fig)
print("saved reports/figures/fig_mde_vs_sample.png")

saved reports/figures/fig_mde_vs_sample.png


In [7]:
# 还需多少天（用 outcome 窗实际累积速度）与 underpowered 结论
extra = {}
for name, req in [("Gross", n_per_group(BASE["p_enroll_given_click"], -LK["mde_gross"], a, pw)),
                  ("Net", n_per_group(BASE["p_pay_given_click"], -LK["mde_net"], a, pw))]:
    nc, ne = actual[name]["nc"], actual[name]["ne"]
    days = LK["windows"]["outcome_days"]
    rate_per_group_day = ((nc+ne)/2)/days
    need_more = (req - (nc+ne)/2)/rate_per_group_day
    extra[name] = dict(required_per_group=round(req), achieved_per_group=round((nc+ne)/2),
                       clicks_per_group_day=round(rate_per_group_day,1),
                       extra_accrual_days=round(need_more,1),
                       extra_calendar_days_with_lag=round(need_more+LK["windows"]["outcome_lag_days"],1))
    print(f"{name}: 每组需 {req:,.0f}，实际 {((nc+ne)/2):,.0f}，按 {rate_per_group_day:.0f} clicks/组/天，"
          f"还需累积 {need_more:.1f} 天（再加14天lag 才成熟）")
print("\n结论：实际样本对锁定 MDE 的功效低于 0.80 → 对 1pp/0.75pp 级别效应 underpowered；")
print("'不显著'不能解读为'无影响'，须结合 CI 宽度与 MDE 解释。")

Gross: 每组需 25,233，实际 17,276，按 751 clicks/组/天，还需累积 10.6 天（再加14天lag 才成熟）
Net: 每组需 26,348，实际 17,276，按 751 clicks/组/天，还需累积 12.1 天（再加14天lag 才成熟）

结论：实际样本对锁定 MDE 的功效低于 0.80 → 对 1pp/0.75pp 级别效应 underpowered；
'不显著'不能解读为'无影响'，须结合 CI 宽度与 MDE 解释。


In [8]:
# 落盘机器可读精度结果 + 回读验证
out = {"locked": {k: LK[k] for k in ["alpha_two_sided","ni_alpha_one_sided","target_power",
                                     "mde_gross","mde_net","ni_delta_net"]},
       "theoretical_requirement": plan,
       "actual_precision": actual,
       "power_by_metric": power_tbl.to_dict(orient="records"),
       "extra_days_needed": extra}
p = ROOT/"data"/"processed"/"design_precision.json"
p.write_text(json.dumps(out, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
back = json.loads(p.read_text(encoding="utf-8"))
assert abs(back["power_by_metric"][0]["power@locked_MDE"] - power_tbl.iloc[0]["power@locked_MDE"]) < 1e-12
print("design_precision.json written & re-read OK")

design_precision.json written & re-read OK


## 小结
1. 参数已事前锁定：α=0.05（NI 单侧 0.025）、power=0.80、MDE Gross=1.0pp / Net=0.75pp、δ=0.75pp（含用户业务理由，见 config）。
2. 理论上每组需约 25.2k（Gross）/26.3k（Net）clicks；实际 outcome 窗每组仅约 17.3k，覆盖率约 66–69%。
3. 实际样本对锁定 MDE 的功效不足 0.80，要 80% 功效需更大的可检测效应或延长实验；存在明确 underpowered / low-precision 风险。
4. 不报告 post-hoc observed power；以上功效均为“在实际样本量下对假设真实效应”的设计式精度。